# Part B - Data Analysis & Testing Tasks: Derivable Judgement

# Importing data and libraries

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import math

data= pd.read_csv('Health_Dataset_CSV.csv')
data.head()

,record_id,age_group,age,weight,gender,region,smoking_status,exercise_frequency,bmi,blood_pressure,diabetes,hypertension,cholesterol_level,glucose_level,visit_date
0,d15f1b71-9a13-4e52-aed7-53c3b0fef6a3,18-25,24,64,Male,North,Non-Smoker,Never,25.8,143.8,True,True,224.1,130.1,2024-12-30
1,d280b34b-d8c9-44f5-9793-7a70f5269f70,18-25,22,77,Female,West,Former Smoker,Rarely,32.7,167.6,True,True,221.9,157.7,2024-08-21
2,3884b275-ee8b-4f88-8725-6741dd497285,36-45,37,48,Female,South,Non-Smoker,Weekly,35.5,156.2,True,True,148.4,156.0,2026-04-26
3,19633c94-6d36-49c8-9973-684a975f2b26,36-45,43,53,Female,North,Former Smoker,Weekly,18.8,122.6,True,False,289.2,132.1,2024-09-11
4,f26c1ae9-db37-4111-ae46-222e7730744e,60+,75,100,Male,West,Smoker,Rarely,34.6,163.8,True,True,269.8,146.4,2025-12-09


# 1. Formulate at least two hypotheses from the dataset. 

## Hypothesis 1: Smoking & Diabetes Prevalence
- **H₀ (Null Hypothesis):** Smoking has no effect on diabetes prevalence.  
- **H₁ (Alternative Hypothesis):** Smoking affects diabetes prevalence.  

**Test Type:**  
- Chi-Square Test of Independence (categorical variables: smoking status vs diabetes status).  

---

## Hypothesis 2: Exercise Frequency & Cholesterol Levels
- **H₀ (Null Hypothesis):** Exercise frequency has no effect on cholesterol levels.  
- **H₁ (Alternative Hypothesis):** Exercise frequency reduces cholesterol levels.  

**Test Type:**  
- ANOVA (if comparing cholesterol across multiple exercise frequency groups).  
- Independent t-test (if comparing only two groups: e.g., regular vs non-regular exercise).  

---

## Hypothesis 3: BMI & Blood Pressure
- **H₀ (Null Hypothesis):** BMI has no effect on blood pressure.  
- **H₁ (Alternative Hypothesis):** Higher BMI increases blood pressure.  

**Test Type:**  
- Correlation/Regression Analysis (continuous variables: BMI vs blood pressure).  


# 2. Calculate Confidence Intervals for key numerical data like age, weight, etc.

In [3]:
# Example: Confidence interval for weight
weights = data['weight'].values

mean_w = np.mean(weights)
std_w = np.std(weights, ddof=1)
n = len(weights)
se = std_w / np.sqrt(n)

z = 1.96  # 95% confidence
lower = mean_w - z*se
upper = mean_w + z*se

print("Mean Weight:", mean_w)
print("95% CI:", (lower, upper))


Mean Weight: 77.43
95% CI: (np.float64(75.7392634770906), np.float64(79.12073652290941))


### Interpretation:
The confidence interval means that we are **95% confident** the true average weight of the population lies between **75.74 kg and 79.12 kg**.  
This range captures the likely population mean based on the sample data.

### Conclusion:
Since the interval is fairly narrow, it suggests the estimate of mean weight is **precise and reliable**.  
We **accept H₀** that the population mean weight is within this range, and conclude that the average weight is approximately **77 kg**, with only minor variation.


# 3. Find the Critical Value and p-value to interpret the test results.


In [4]:
from scipy.stats import norm

# Compare BMI of smokers vs non-smokers
smokers = data[data['smoking_status']=="Smoker"]['bmi'].values
non_smokers = data[data['smoking_status']=="Non-Smoker"]['bmi'].values

mean1, mean2 = np.mean(smokers), np.mean(non_smokers)
std1, std2 = np.std(smokers, ddof=1), np.std(non_smokers, ddof=1)
n1, n2 = len(smokers), len(non_smokers)

z_stat = (mean1 - mean2) / np.sqrt((std1**2/n1) + (std2**2/n2))
p_value = 2 * (1 - norm.cdf(abs(z_stat)))
alpha = 0.05  # significance level
critical_value = norm.ppf(1 - alpha/2)  


print("z-stat:", z_stat)
print("p-value:", p_value)
print("Critical Value:", critical_value)


z-stat: 0.33132027517418333
p-value: 0.7404025779917842
Critical Value: 1.959963984540054


### Interpretation of Z-Test (Smokers vs Non-Smokers BMI)

- **z-statistic = 0.33**
- **Critical Value (95%) = 1.96**
- **p-value = 0.74**

### Conclusion:
Since the z-statistic (0.33) is **less than** the critical value (1.96) and the p-value (0.74) is **greater than 0.05**, we **fail to reject H₀**.  
This means there is **no statistically significant difference** in the mean BMI between smokers and non-smokers in this dataset.

# 4. Perform z-test or t-test based on sample size (mean comparison across groups).

comparing BMI of smokers vs non‑smokers (Same as above)

In [16]:
from math import sqrt

# Separate groups
group1 = data[data['smoking_status']=="Smoker"]['bmi'].values
group2 = data[data['smoking_status']=="Non-Smoker"]['bmi'].values

mean1, mean2 = np.mean(group1), np.mean(group2)
std1, std2 = np.std(group1, ddof=1), np.std(group2, ddof=1)
n1, n2 = len(group1), len(group2)
var1, var2= np.var(group1), np.var(group2)

mean1, mean2
# we have mean > 30 so we use T-test

var1,var2
# variance are not

# T-test (Welch’s t)
t_test = (mean1 - mean2) / sqrt((std1**2/n1) + (std2**2/n2))

# Degrees of freedom (Welch’s approximation)
df = ((std1**2/n1 + std2**2/n2)**2) / (
    ((std1**2/n1)**2/(n1-1)) + ((std2**2/n2)**2/(n2-1)))

critical_value = stats.t.ppf(1 - alpha/2, df) 
p_value = 2 * (1 - stats.t.cdf(abs(t_test), df))

print("Mean1 (Smokers):", mean1)
print("Mean2 (Non-Smokers):", mean2)
print("Test Statistic:", t_test)
print("Critical Value (95%):", critical_value)
print("p-value:", p_value)

Mean1 (Smokers): 28.087421383647797
Mean2 (Non-Smokers): 27.872839506172838
Test Statistic: 0.33132027517418333
Critical Value (95%): 1.967430678111727
p-value: 0.7406201891935213


### Interpretation of Welch’s T-Test (Smokers vs Non-Smokers BMI)

- **Mean BMI (Smokers):** 28.09  
- **Mean BMI (Non-Smokers):** 27.42  
- **T-test Statistic:** 1.04  
- **Critical Value (95%):** 1.96  
- **p-value:** 0.30  

### Conclusion:
Since the T-test statistic (1.04) is **less than** the critical value (1.96) and the p-value (0.30) is **greater than 0.05**, we **fail to reject H₀**.  
This means there is **no statistically significant difference** in the mean BMI between smokers and non-smokers in this dataset.


# 5. Conduct a chi-square test on categorical data (e.g., Smoking habit vs Disease).


In [6]:
from scipy import stats

# Contingency Table Smoking habit vs Diabetes

contingency = pd.crosstab(data['smoking_status'], data['diabetes'])
observed = contingency.values

# Row Totals, Column Totals, Grand Total

row_totals = observed.sum(axis=1)
col_totals = observed.sum(axis=0)
grand_total = observed.sum()

# Expected Frequencies 
# E = (Row Total × Column Total) / Grand Total

expected = np.outer(row_totals, col_totals) / grand_total

# Chi-Square Statistic 
# χ² = Σ (O - E)^2 / E

chi_square_stat = ((observed - expected) ** 2 / expected).sum()

# Degrees of Freedom
# df = (rows - 1)(columns - 1)

r, c = observed.shape
df_chi = (r - 1) * (c - 1)

# Critical Value & p-value

alpha = 0.05
critical_value = stats.chi2.ppf(1 - alpha, df_chi)
p_value = 1 - stats.chi2.cdf(chi_square_stat, df_chi)

# Decision making

if chi_square_stat > critical_value:
    decision = "Reject H₀ (Smoking and Diabetes are related)"
else:
    decision = "Fail to Reject H₀ (Smoking and Diabetes are independent)"

# Final Output

print("Observed Table:\n", contingency)
print("\nExpected Table:\n", expected)
print("\nChi-Square Statistic:", chi_square_stat)
print("Degrees of Freedom:", df_chi)
print("Critical Value (95%):", critical_value)
print("p-value:", p_value)
print("Decision:", decision)

Observed Table:
 diabetes        False  True 
smoking_status              
Former Smoker      31    148
Non-Smoker         17    145
Smoker             18    141

Expected Table:
 [[ 23.628 155.372]
 [ 21.384 140.616]
 [ 20.988 138.012]]

Chi-Square Statistic: 4.175408027330655
Degrees of Freedom: 2
Critical Value (95%): 5.991464547107979
p-value: 0.1239714460523772
Decision: Fail to Reject H₀ (Smoking and Diabetes are independent)


### Interpretation of Chi-Square Test (Smoking vs Diabetes)

- **Chi-Square Statistic:** 4.16  
- **Degrees of Freedom:** 2  
- **Critical Value (95%):** 5.99  
- **p-value:** 0.124  

### Conclusion:
Since the Chi-Square statistic (4.16) is **less than** the critical value (5.99) and the p-value (0.124) is **greater than 0.05**, we **fail to reject H₀**.  
This means that **smoking status and diabetes are independent** in this dataset — there is **no statistically significant relationship** between smoking habit and diabetes occurrence.


# 6. Perform an ANOVA test to check if age groups significantly differ in disease rate.

In [27]:
# Step 1: Group disease rates by age_group
groups = []
for age_group, subset in data.groupby('age_group'):
    disease_rate = subset['diabetes'].astype(int).values
    groups.append(disease_rate)

# Step 2: Grand mean
all_data = [x for g in groups for x in g]
grand_mean = sum(all_data) / len(all_data)

# Step 3: Between-group sum of squares (SSB)
ss_between = sum(len(g) * (sum(g)/len(g) - grand_mean)**2 for g in groups)

# Step 4: Within-group sum of squares (SSW)
ss_within = sum(sum((x - (sum(g)/len(g)))**2 for x in g) for g in groups)

# Step 5: Degrees of freedom
df_between = len(groups) - 1
df_within = len(all_data) - len(groups)

# Step 6: Mean squares
ms_between = ss_between / df_between
ms_within = ss_within / df_within

# Step 7: F-statistic
F_test = ms_between / ms_within

# Step 8: Critical value & p-value
alpha = 0.05
critical_value = stats.f.ppf(1 - alpha, df_between, df_within)
p_value = 1 - stats.f.cdf(F_stat, df_between, df_within)

print("Grand Mean:", grand_mean)
print("SS Between:", ss_between)
print("SS Within:", ss_within)
print("F-Test:", F_test)
print("Degrees of Freedom:", (df_between, df_within))
print("Critical Value (95%):", critical_value)
print("p-value:", p_value)

Grand Mean: 0.868
SS Between: 0.44896890143988194
SS Within: 56.83903109856006
F-Test: 0.977495577939099
Degrees of Freedom: (4, 495)
Critical Value (95%): 2.389947844458197
p-value: 0.4193652067476029


### Interpretation of ANOVA Test (Age Groups vs Disease Rate)

- **F-Test Statistic:** 0.98  
- **Critical Value (95%):** 2.39  
- **p-value:** 0.42  

### Conclusion:
Since the F-Test statistic (0.98) is **less than** the critical value (2.39) and the p-value (0.42) is **greater than 0.05**, we **fail to reject H₀**.  
This means that **disease rates do not differ significantly across age groups** in this dataset.  
In other words, age group does **not appear to be a significant factor** influencing disease occurrence here.


# 7. Calculate Covariance and Correlation between continuous variables (e.g., Age vs. BMI).

In [29]:
# Select variables
age = data['age']
bmi = data['bmi']

# --- Covariance matrix ---
cov_matrix = data[['age', 'bmi']].cov()
cov_ab = cov_matrix.loc['age', 'bmi']

# --- Correlation matrix ---
corr_matrix = data[['age', 'bmi']].corr()
corr_ab = corr_matrix.loc['age', 'bmi']

print("Covariance (Age vs BMI):", cov_ab)
print("Correlation (Age vs BMI):", corr_ab)

Covariance (Age vs BMI): -1.3424080160320673
Correlation (Age vs BMI): -0.012831401818982893


### Interpretation of Covariance and Correlation (Age vs BMI)

- **Covariance:** −1.34  
- **Correlation:** −0.0128  

### Conclusion:
The negative covariance suggests a very slight tendency for BMI to decrease as Age increases.  
However, the correlation coefficient is extremely close to **0**, indicating **almost no linear relationship** between Age and BMI.  

This means Age and BMI are essentially **independent variables** in this dataset, with negligible association.
